# 02 — Analysis: statistics, figures, results table

No CARLA dependency — this notebook only reads `results/raw_episodes.csv`, so it can be run on Colab right after the experiment grid, or downloaded and run on any machine with `numpy`, `pandas`, `scipy`, `matplotlib` (this machine included).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "../src")  # or "/content/fag-project/src" on Colab

import pandas as pd

from analysis.stats import summarize, kruskal_wallis_by_density, pairwise_mannwhitney, bonferroni_correct, DEFAULT_METRICS
from analysis.plots import generate_all_figures
from analysis.report import generate_report

RESULTS_CSV = Path("../results/raw_episodes.csv")
df = pd.read_csv(RESULTS_CSV)
print(f"{len(df)} episodes loaded")
df.head()

## Summary table (bootstrap mean + 95% CI per strategy x density)

In [ ]:
summary = summarize(df)
summary

## Overall strategy effect per density (Kruskal-Wallis)

Run this before pairwise tests — only dig into pairwise comparisons for metrics where the overall test is significant, to avoid fishing.

In [ ]:
for metric in DEFAULT_METRICS:
    print(kruskal_wallis_by_density(df, metric))

## Pairwise Mann-Whitney (only for metrics found significant above)

In [ ]:
pw = pairwise_mannwhitney(df, "min_ttc_s", density="medium")
pw["p_value_bonferroni"] = bonferroni_correct(pw["p_value"])
pw

## Figures

In [ ]:
generate_all_figures(df, Path("../results/figures"))
print("Figures written to results/figures/")

## Failure-mode breakdown (raw counts)

In [ ]:
pd.crosstab(df["strategy"], df["failure_mode"])

## Full markdown report

In [ ]:
generate_report(RESULTS_CSV, Path("../results/report.md"))
print(Path("../results/report.md").read_text())